# Miner Tips 03: Layerwise Distillation

Layerwise distillation means you do not ask a compressed model to learn everything at once.

Instead:

```text
teacher hidden input
        |
        v
student layer N tries to match teacher layer N output
        |
        v
freeze layer N
        |
        v
move to layer N+1
```

This is useful for both binary and ternary models because low-bit layers can drift. Matching layer outputs keeps the student close before you run full text PPL.

In [ ]:
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ImportError:
    torch = None
    nn = None
    F = None

print("torch available:", torch is not None)

In [ ]:
if torch is not None:
    torch.manual_seed(0)

    class TinyBlock(nn.Module):
        def __init__(self, width):
            super().__init__()
            self.fc1 = nn.Linear(width, width)
            self.fc2 = nn.Linear(width, width)

        def forward(self, x):
            return x + self.fc2(torch.tanh(self.fc1(x)))

    class TinyStack(nn.Module):
        def __init__(self, width=16, layers=3):
            super().__init__()
            self.blocks = nn.ModuleList([TinyBlock(width) for _ in range(layers)])

        def forward(self, x, stop_after=None):
            for i, block in enumerate(self.blocks):
                x = block(x)
                if stop_after is not None and i == stop_after:
                    break
            return x

    teacher = TinyStack()
    student = TinyStack()
    student.load_state_dict(teacher.state_dict())
    print("tiny teacher/student created")
else:
    print("Install torch to run the toy distillation cells.")

In [ ]:
def ternary_weight_(param):
    """In-place toy ternary projection.

    Real recipes usually use better scales, grouping, and rescue rows.
    """
    with torch.no_grad():
        scale = param.abs().mean().clamp_min(1e-8)
        threshold = 0.7 * scale
        q = torch.sign(param) * scale
        q = torch.where(param.abs() >= threshold, q, torch.zeros_like(q))
        param.copy_(q)


def quantize_one_block_ternary(block):
    for name, param in block.named_parameters():
        if "weight" in name:
            ternary_weight_(param)


def distill_one_block(teacher, student, block_index, steps=80, lr=2e-2):
    """Train one student block to match the teacher block output.

    Earlier student blocks are used to create the input. Then only the current
    block's parameters get updated.
    """
    for p in student.parameters():
        p.requires_grad_(False)
    for p in student.blocks[block_index].parameters():
        p.requires_grad_(True)

    opt = torch.optim.AdamW(student.blocks[block_index].parameters(), lr=lr)

    for step in range(steps):
        x0 = torch.randn(32, 16)
        with torch.no_grad():
            # Input seen by this layer after the already-compressed prefix.
            student_prefix = x0 if block_index == 0 else student(x0, stop_after=block_index - 1)
            teacher_target = teacher(x0, stop_after=block_index)

        pred = student.blocks[block_index](student_prefix)
        loss = F.mse_loss(pred, teacher_target)
        opt.zero_grad()
        loss.backward()
        opt.step()

    return float(loss.detach())

In [ ]:
if torch is not None:
    losses = []
    for layer in range(len(student.blocks)):
        quantize_one_block_ternary(student.blocks[layer])
        loss = distill_one_block(teacher, student, layer)
        losses.append(loss)
        print(f"layer {layer}: final fit mse = {loss:.6f}")

    x = torch.randn(128, 16)
    with torch.no_grad():
        final_mse = F.mse_loss(student(x), teacher(x)).item()
    print("full-stack teacher/student mse:", round(final_mse, 6))
else:
    print("Install torch to run the toy distillation loop.")

## Real-model translation

Use the same structure with Qwen blocks:

1. Choose a public calibration text set.
2. Cache teacher hidden states or compute them batch-by-batch.
3. Quantize one student block.
4. Train only that block to match the teacher block output.
5. Freeze it and move forward.
6. After all blocks, run public/dev Qwen-token PPL.

Layer MSE is not the leaderboard metric. It is a training signal that should improve text PPL when the calibration text is representative.